# This notebook is history, not the working code

The safety scores used to be calculated in this notebook. The scores are now calculated by
the `ridescore` package in `src/ridescore/`, and that package is the only thing that runs.
Changing a scoring rule means changing the package, not this notebook.

To produce a set of scores and look at them:

    uv sync
    uv run ridescore run
    uv run ridescore inspect

## Why this notebook is kept

The package was written to reproduce exactly what this notebook did — including several
things this notebook got wrong. Those mistakes were copied deliberately, so that each one
can be found, discussed and corrected on its own. Correcting them quietly, all at once,
would have moved every score on the map with no way to tell which change caused what.

Each copied mistake is marked `DEFECT` in `src/ridescore/config.py`. Deciding whether a
correction to one of those mistakes is right means comparing the new behaviour against the
old behaviour, and this notebook is the only record of the old behaviour.

## Please do not edit this notebook

Editing this notebook changes nothing that anyone sees: no map, no score and no published
data depends on this notebook. An edit would only make this notebook stop matching the code
that was written from it — and that match is the one thing this notebook is still here for.

| Where to look | What you find there |
|---|---|
| `docs/running-the-pipeline.md` | how to run the package |
| `docs/ported-defects.md` | the copied mistakes, and why each one was kept |
| `notebooks/plot_run_output.ipynb` | charts and a map of what a run produced |

# DC Bike Safety Map: Data Processing

This notebook takes [road](https://opendata.dc.gov/datasets/DCGIS::roadway-block/about) and [crash](https://opendata.dc.gov/datasets/crashes-in-dc/about) data from the DC data website and processes it for the [DC Bike Safety Map](http://161.35.142.176/). The LTS and ridescore build on 01_lts_osm_elia_v2.ipynb.

This is preliminary work and future work could include:
* see if same data can be extracted from OSM
* Taking directionality of road into account
* Adding bike trails
* Implement standard LTS methodology
* Refine scaling factors to 0 to 100 scale

In [ ]:
# 1) Imports + settings
import warnings, math, re, os, json

import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point
import osmnx as ox
import folium
from folium.plugins import MeasureControl
import requests
import matplotlib.pyplot as plt
import numpy as np

import requests

import time
from datetime import date
from dateutil.relativedelta import relativedelta

ox.settings.use_cache = True
ox.settings.log_console = False

print("Versions →", 
      "osmnx", ox.__version__, 
      "| geopandas", gpd.__version__)

## Road Data

In [ ]:
# 2) Get Roadblock data from website
CRS = 'EPSG:4326'
def try_fetch_dc_roads():
    url = "https://opendata.arcgis.com/datasets/DCGIS::roadway-block.geojson"
    try:
        print('trying website')
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        print(r)
        print(url)
        if r.ok:
            gj = r.json()
            roads = gpd.GeoDataFrame.from_features(gj["features"], crs=CRS)
            return roads
    except Exception as error:
        print('Error reading road data from dc gov website')

edges = try_fetch_dc_roads()
edges.to_file('edges.geojson')
edges

In [ ]:
#look at columns
for c in gdf.columns:
    print(c)

In [ ]:
# 3) Helpers for tag parsing
def classify_facility(tags):
    # standardize bike lane types into 3 categories (protected, buffered, and painted). Sharrows don't seem to be indicated in dataset
    # directionality of road is not taken into account
    if (tags.get("BIKELANE_PROTECTED") in {"IB", "OB", "BD"}) or (tags.get("BIKELANE_DUAL_PROTECTED") in {"IB", "OB", "BD"}):
        return "protected_track"
    elif (tags.get("BIKELANE_BUFFERED") in {"IB", "OB", "BD"}):
        return "buffered_lane"
    elif (tags.get("BIKELANE_CONVENTIONAL") in {"IB", "OB", "BD"}):
        return "painted_lane"
    else:
        return "none"

#replace function number with name
function_dict = {11: 'Interstate', 12: 'Other Freeway and Expressway', 14:  'Principal/Primary Arterial', 16:  'Minor Arterial', 17:  'Collector', 19:'Local'}
def name_function(tags):
    if tags['DCFUNCTIONALCLASS'] in function_dict.keys():
        return function_dict[tags['DCFUNCTIONALCLASS']]
    else:
        return 'Other'



In [ ]:
# 3) simplify and impute fields as needed.
# while the safety scores my included the impute values to ensure it processes without error, the raw values are displayed on the map.

rows = []
for i, r in edges.reset_index(drop=True).iterrows():
    if (i%1000) == 0:
        print(i)
    tags = r.to_dict()
    facility = classify_facility(tags)
    function = name_function(tags)
    rows.append({
        "route_id": tags['ROUTEID'],
        "route_name": tags['ROUTENAME'],
        "function": function,
        "num_lanes": tags['TOTALTRAVELLANES'] if tags['TOTALTRAVELLANES'] >-1 else 1, #set number of travels to 1 if no value is available 
        "num_lanes_raw": tags['TOTALTRAVELLANES'],
        "speed_limit": tags['SPEEDLIMITS_OB'] if tags['SPEEDLIMITS_OB'] >1 else 25, #speed limit is set to 25 is no values is available
        "speed_limit_raw": tags['SPEEDLIMITS_OB'],
        "bike_facility_type": facility,
        "parking_presence": True if tags["TOTALPARKINGLANES"] > 0 else False,
        "road_width": tags['TOTALCROSSSECTIONWIDTH'],
        "slow_street": tags['SLOWSTREETINFO'],
        "pavement_condition": tags['PCI_CONDCATEGORY'],
        "geometry": r.geometry
    })

gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)
print("Normalized segments:", len(gdf))
gdf.head(2)

### Modified Level of Traffic Stress score

We use our own, modified level of traffic stress (LTS) calculator.

<table>
  <tbody>
    <tr>
      <th>Bike lane</th>
      <th>Number of lanes</th>
      <th>Speed limit</th>
      <th>Road function</th>
      <th>LTS</th>
    </tr>
    <tr style="background-color: #a4f1b6;">
      <td>Protected track</td>
      <td>-</td>
      <td>-</td>
      <td>-</td>
      <td>1</td>
    <tr style="background-color: #f7f08c;">
      <td>Buffered lane or painted lane</td>
      <td>&lt;= 2</td>
      <td>&lt;= 25</td>
      <td>-</td>
      <td>2</td>
    </tr>
    <tr style="background-color: #f7f08c;">
      <td>None</td>
      <td>&lt;= 2</td>
      <td>&lt;= 25</td>
      <td>Local</td>
      <td>2</td>
    </tr>
    <tr style="background-color: #f0c77b;">
      <td>Buffered lane or painted lane</td>
      <td>&gt;2 and &lt;=3</td>
      <td>&gt;25 and &lt;= 30</td>
      <td>-</td>
      <td>3</td>
    </tr>
    <tr style="background-color: #f0c77b;">
      <td>None</td>
      <td>&lt;=2</td>
      <td>&gt;25 and &lt;= 30</td>
      <td>Local</td>
      <td>3</td>
    </tr>
    <tr style="background-color: #e47d86;">
      <td colspan="4">Any other combination</td>
      <td>4</td>
    </tr>
  </tbody>
</table>

In [ ]:
# 4) LTS rules (compact + tunable)

def lts_level(facility, speed, lanes, function=""):
    hw = (function or "").lower()
    if facility in {"protected_track"}: return 1 # or hw in {"cycleway","path"}: return 1, also include ,"separated_lane" here
    if facility in {"buffered_lane","painted_lane"}:
        if speed <= 25 and lanes <= 2: return 2
        if speed <= 30 and lanes <= 3: return 3
        return 4
    if facility in {"none"}: #include "shared" here
        if speed <= 20 and lanes <= 2 and hw in {"local"}: 
            return 2
        if speed <= 30 and lanes <= 2 and hw in {"local"}: return 3
        return 4
    return 4

gdf["lts_level"] = gdf.apply(lambda r: lts_level(r.bike_facility_type, int(r.speed_limit), int(r.num_lanes), r.function), axis=1)
gdf["lts_level"].value_counts().sort_index()

### Data exploration
Let's look at some of the columns of interest

In [ ]:
plt.hist(gdf['lts_level'])
plt.xlabel('level of stress')
plt.ylabel('count')
plt.title('Histogram of lts levels')
plt.show()

In [ ]:
#change crs to get length in meters
gdf = gdf.to_crs(gdf.estimate_utm_crs())
gdf['len'] = gdf.length
print("length of road network: {:.2f}km".format(sum(gdf['len'])/1000))

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 15))


sp = gdf.groupby('lts_level', dropna=False)['len'].sum()/gdf['len'].sum()*100
sp.plot(kind='bar', title = 'Percent of read network with lts level', ylabel = 'percent', ax=axes[0,0])

sp = gdf.groupby('speed_limit_raw', dropna=False)['len'].sum()/gdf['len'].sum()*100
sp.plot(kind='bar', title = 'Percent of read network with each speed limit', ylabel = 'percent', ax=axes[0,1])

sp = gdf.groupby('num_lanes_raw', dropna=False)['len'].sum()/gdf['len'].sum()*100
sp.plot(kind='bar', title = 'Percent of read network with each number of lanes', ylabel = 'percent', ax=axes[1,0])

sp = gdf.groupby('function', dropna=False)['len'].sum()/gdf['len'].sum()*100
sp.plot(kind='bar', title = 'Percent of read network with each speed limit', ylabel = 'percent', ax=axes[1,1])

sp = gdf.groupby('bike_facility_type', dropna=False)['len'].sum()/gdf['len'].sum()*100
sp.plot(kind='bar', title = 'Percent of read network with each bike lane type', ylabel = 'percent', ax=axes[2,0])

sp = gdf.groupby('pavement_condition', dropna=False)['len'].sum()/gdf['len'].sum()*100
sp.plot(kind='bar', title = 'Percent of read network with each pavement condition', ylabel = 'percent', ax=axes[2,1])

plt.tight_layout()
plt.show()

In [ ]:
gdf.to_file('levels.geojson')

## Get Crash Data

We only use crashes that resulted in a bicyclist fatality or injury from the last 5 years.

In [ ]:
def get_crashes_df(start_date, end_date, chunk_size=1000):
    base_url = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Public_Safety_WebMercator/MapServer/24/query"

    all_rows = []
    offset = 0

    while True:
        params = {
            "where": (
                f"REPORTDATE >= DATE '{start_date} 00:00:00' "
                f"AND REPORTDATE <= DATE '{end_date} 23:59:59' "
                "AND (MAJORINJURIES_BICYCLIST > 0 "
                "OR MINORINJURIES_BICYCLIST > 0"
                "OR UNKNOWNINJURIES_BICYCLIST > 0"
                "OR FATAL_BICYCLIST > 0)"
            ),
            "outFields": "*",
            "outSR": 4326,
            "f": "json",
            "orderByFields": "OBJECTID",
            "resultOffset": offset,
            "resultRecordCount": chunk_size
        }

        r = requests.get(base_url, params=params, timeout=120)
        data = r.json()

        features = data.get("features", [])

        #print(f"Received {len(features)} features")

        # Extract attribute dictionaries
        for feature in features:
            attrs = feature.get("attributes", {})
            geom = feature.get("geometry")

            if geom and "x" in geom and "y" in geom:
                attrs["geometry"] = Point(geom["x"], geom["y"])
                all_rows.append(attrs)

        # Stop when last page is reached
        if len(features) < chunk_size:
            break

        offset += chunk_size
        print(f"Fetched {offset} records...")
        time.sleep(0.05)

    # Convert to GeoDataFrame
    df = pd.DataFrame(all_rows)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

    return gdf
    
start_date = (date.today()-relativedelta(years=5)).isoformat()
print(start_date)
end_date = date.today().isoformat()
print(end_date)
crashes_new = get_crashes_df(start_date, end_date)
print(len(crashes_new))

In [ ]:
# restrict crashes to DC (probably not necessary considering they are coming from DC website)

PLACE_NAME = "Washington, District of Columbia, USA"
CRS = "EPSG:4326"

# Crash join toggles (turn off if ArcGIS blocks)
CRASH_ENABLE = True
YEARS_BACK = 5
CRASH_BUFFER_M = 10

# 3) Area of interest
aoi_gdf = ox.geocode_to_gdf(PLACE_NAME).to_crs(CRS)
aoi = aoi_gdf.geometry.iloc[0]
aoi_gdf

In [ ]:
crashes_new = gpd.overlay(crashes_new, aoi_gdf[["geometry"]], how="intersection")
crashes_new['REPORTDATE_'] = pd.to_datetime(crashes_new["REPORTDATE"], unit="ms", utc=True).dt.tz_convert("America/New_York")
crashes_new

In [ ]:
crashes_new.to_file('crashes_dcdata.geojson')

In [ ]:
keep_columns = ['REPORTDATE_', 'ADDRESS', 'MAJORINJURIES_BICYCLIST', 'MINORINJURIES_BICYCLIST', 'UNKNOWNINJURIES_BICYCLIST','FATAL_BICYCLIST', 'TOTAL_BICYCLES', 'BICYCLISTSIMPAIRED', 'TOTAL_VEHICLES', 'TOTAL_PEDESTRIANS', 'geometry']
crashes_reduced = crashes_new[keep_columns]
crashes_reduced.to_file('crashes_reduced.geojson')

In [ ]:
print('number of injuries and fatalities in the last 5 years:')
inj = ['MAJORINJURIES_BICYCLIST', 'MINORINJURIES_BICYCLIST', 'UNKNOWNINJURIES_BICYCLIST','FATAL_BICYCLIST']
total= 0
for i in inj:
    print(i+': '+str(sum(crashes_reduced[i])))
    total = total + sum(crashes_reduced[i])
print('total: '+str(total))


In [ ]:
# 9) Crash counts → segments (buffered spatial join; handles no-spatial-index case)

def count_crashes_near_segments(segments_gdf, crashes_gdf, buffer_m=10):
    if crashes_gdf.empty or segments_gdf.empty:
        segments_gdf["crash_count_5yr"] = 0
        return segments_gdf

    proj = "EPSG:3857"  # meters
    seg_p = segments_gdf.to_crs(proj).copy()
    cr_p  = crashes_gdf.to_crs(proj).copy()

    seg_p["buf"] = seg_p.geometry.buffer(buffer_m)
    seg_p = seg_p.set_geometry("buf", crs=proj)

    try:
        print('joined')
        joined = gpd.sjoin(cr_p[["geometry"]], seg_p[["buf"]], how="left", predicate="within") #this will create multiple matches
        print('length of joined: '+str(len(joined)))
    except Exception as e:
        # If spatial index missing, geopandas prints an rtree/pygeos message—fallback to brute force (slow but safe for MVP)
        from shapely.prepared import prep
        counts = []
        prepped = [prep(g) for g in seg_p["buf"]]
        for pt in cr_p.geometry:
            hits = [i for i, pg in enumerate(prepped) if pg.contains(pt)]
            counts.extend(hits)
        joined = pd.Series(counts).value_counts().rename_axis("index_right").rename("size").to_frame()

        seg_p["crash_count_5yr"] = 0
        seg_p.loc[joined.index, "crash_count_5yr"] = joined["size"].values
        return seg_p.drop(columns=["buf"]).set_geometry("geometry").to_crs(segments_gdf.crs)

    counts = joined.groupby(joined.index_right).size().rename("crash_count_5yr")
    seg_p = seg_p.drop(columns=["buf"]).set_geometry("geometry")
    seg_p["crash_count_5yr"] = counts.reindex(seg_p.index).fillna(0).astype(int)
    return seg_p.to_crs(segments_gdf.crs)

gdf = count_crashes_near_segments(gdf, crashes_new, buffer_m=CRASH_BUFFER_M)
gdf["serious_injury_count_5yr"] = 0
gdf["fatal_count_5yr"] = 0
gdf[["crash_count_5yr"]].describe()

In [ ]:
print('segments with most crashes')
gdf[gdf['crash_count_5yr']>0].sort_values(by='crash_count_5yr', ascending = False)

In [ ]:
# 10) RideScore (0–100)

def p95(values):
    s = sorted([int(v) for v in values if pd.notnull(v)])
    if not s: return 1
    k = int(round(0.95 * (len(s) - 1)))
    return max(s[k], 1)

def lts_to_score(x): return {1:100, 2:75, 3:40, 4:10}.get(int(x), 10)

facility_bonus = {"protected_track":10, "separated_lane":10, "buffered_lane":5, "painted_lane":3, "shared":0, "none":0}

P95_CRASH = p95(gdf["crash_count_5yr"])
def crash_inv(n): return 100.0 * (1.0 - min(max(float(n)/float(P95_CRASH), 0.0), 1.0))

W_LTS, W_CRASH, W_FAC = 0.6, 0.3, 0.1

gdf["s_LTS"] = gdf["lts_level"].map(lts_to_score)
gdf["s_crash"] = gdf["crash_count_5yr"].map(crash_inv)
gdf["s_facility"] = gdf["bike_facility_type"].map(facility_bonus).fillna(0)
gdf["ridescore_v1"] = (W_LTS*gdf.s_LTS + W_CRASH*gdf.s_crash + W_FAC*gdf.s_facility).round(1)

gdf[["s_LTS","s_crash","s_facility","ridescore_v1"]].describe().round(1)

In [ ]:
plt.hist(gdf['ridescore_v1'])

In [ ]:
gdf.to_file('gdf_withScore_dcdata.geojson')

In [ ]:
#value to scale

#lts to lts_score (already done)
gdf['lts_score'] = gdf['s_LTS']

#speed limit (if no value- speed is 25- too positive?)
def speedlimit_to_score(x):
    return 100-(2*x)
gdf["speedlimit_score"] = gdf["speed_limit"].map(speedlimit_to_score)

#number of lanes, blank goes to 10 score
def num_lanes_to_score(x): return {0:100, 1:100, 2:75, 3:50, 4:25, 5:25, 6:0, 7:0, 8:0, 10:0}.get(int(x), 10)
gdf["num_lanes_score"] = gdf["num_lanes_raw"].map(num_lanes_to_score)

#bike lanes
def facility_to_score(x): return {'protected_track':100, 'buffered_lane':75, 'painted_lane': 50, 'none': 0}.get(x, 0)
gdf["facility_score"] = gdf["bike_facility_type"].map(facility_to_score)

#road type
def function_to_score(x): return {'Local': 100, 'Collector': 75, 'Minor Arterial':50, 
       'Principal/Primary Arterial':25, 'Other Freeway and Expressway': 0,
       'Interstate':0, 'Other':0}.get(x, 0)
gdf["function_score"] = gdf["function"].map(function_to_score)

#road width
def road_width_to_score(x):
    return 100-x
gdf["road_width_score"] = gdf["road_width"].map(road_width_to_score)

#slow street- seems to have ended in 2021

#pavement condition
def pavement_condition_to_score(x): return {'Excellent': 100, 'Good': 75, 'Fair':50, 
       'Poor':25,'Very Poor': 0}.get(x, 50)
gdf["pavement_condition_score"] = gdf["pavement_condition"].map(pavement_condition_to_score)

gdf[['pavement_condition', 'pavement_condition_score']]

In [ ]:
# 10A) Build a lean render GeoDataFrame

# === knobs you can tweak ===
SIMPLIFY_TOL_M   = 4      # simplify by ~4 meters
ROUND_COORDS     = 5      # round lon/lat (~1 m)
MIN_LENGTH_M     = 8      # drop tiny segments
KEEP_FIELDS      = gdf.columns       # keep all
#["crash_count_5yr","parking_presence","num_lanes_raw", "speed_limit_raw", "bike_facility_type", "route_id", "lts_level", "ridescore_v1", "geometry"]
#["segment_id", "lts_level", "ridescore_v1", "geometry"]
FILTER_SCORE_MIN = None   # e.g., 20
FILTER_SCORE_MAX = None   # e.g., 80

# Start from the scored 'gdf' produced in cell 10
render = gdf[KEEP_FIELDS].copy()

# Optional score filters
if FILTER_SCORE_MIN is not None:
    render = render[render["ridescore_v1"] >= float(FILTER_SCORE_MIN)]
if FILTER_SCORE_MAX is not None:
    render = render[render["ridescore_v1"] <= float(FILTER_SCORE_MAX)]

# Drop duplicate geometries
render["_wkb"] = render.geometry.apply(lambda geom: geom.wkb)
render = render.drop_duplicates("_wkb").drop(columns="_wkb")

# Drop very short segments
rp = render.to_crs(3857)
render = render[rp.length >= MIN_LENGTH_M].copy()

# Simplify then return to WGS84
rp = render.to_crs(3857)
rp["geometry"] = rp.geometry.simplify(SIMPLIFY_TOL_M, preserve_topology=True)
render = rp.to_crs(4326)

# Round coordinates
from shapely.geometry import LineString
def round_linestring(ls, n=ROUND_COORDS):
    #print(ls)
    coords = [(round(x, n), round(y, n), z) for x, y, z in ls.coords]
    return LineString(coords)

render["geometry"] = render.geometry.apply(lambda g: round_linestring(g, ROUND_COORDS))
print("Render features:", len(render))
render.head(2)

In [ ]:
render['parking_presence'] = render['parking_presence'].astype(int)

In [ ]:
render.to_file('render_dcdata_wCrashes.geojson')

In [ ]:
render_wkt = render.copy()
render_wkt = render_wkt.drop(columns = 'geometry')
render_wkt.to_csv('render_dc_data_wCrashes.csv')